#Initialization

In [0]:
import pyspark.sql.functions as F

#Read Bronze Table

In [0]:
procedures = spark.table("health.bronze.procedures")

#Data Understanding

###Review Schema

In [0]:
procedures.printSchema()
display(procedures.limit(10))

#Data Profiling

In [0]:
display(
    procedures.select(
        [
            F.sum(
                F.when(F.col(c).isNull(), 1).otherwise(0)
            ).alias(c)
            for c in procedures.columns
        ]
    )
)

###Business Key Validation

In [0]:
total_rows = procedures.count()

distinct_procedure_ids = (
    procedures
    .select("procedure_id")
    .distinct()
    .count()
)

print(f"Total Rows: {total_rows}")
print(f"Distinct Procedure IDs: {distinct_procedure_ids}")

###Cost Validation


In [0]:
display(
    procedures.select(
        F.min("procedure_cost").alias("min_procedure_cost"),
        F.max("procedure_cost").alias("max_procedure_cost")
    )
)

###Duplicate Validation

In [0]:
print(f"Total Rows: {procedures.count()}")

print(
    f"Rows After dropDuplicates(): {procedures.dropDuplicates().count()}"
)

###Remove Duplicate Records

In [0]:
procedures_clean = procedures.dropDuplicates()

###Validation Summary

In [0]:
print(f"Clean Procedures: {procedures_clean.count()}")
print(f"Original Procedures: {procedures.count()}")

#Write Silver Tables

In [0]:
(
    procedures_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("health.silver.procedures")
)

###Verify Silver Procedures

In [0]:
display(
    spark.table("health.silver.procedures")
    .limit(10)
)